# Мини-проект: анализ данных — Трек B (WalletApp)

**Задача от заказчика:** пользователей приходит много, а до оплаты доходит непропорционально мало.
Нужно найти, на каком этапе воронки теряются деньги, и почему, а не просто зафиксировать факт.

**Гипотезы, которые проверяем:**
1. Влияет ли канал привлечения на конверсию в платящего пользователя и на стоимость привлечения.
2. Объясняется ли провал на этапе KYC составом аудитории (возраст/устройство/регион/период) — или это системная проблема, не зависящая от того, кто пользователь.

Ниже — разведка данных, воронка, обе гипотезы с проверкой, статзначимость, оценка денег на кону и итоговые рекомендации.

In [1]:
import os, getpass

# Данные для подключения не хранятся в файле: введи их при запуске
LOGIN = os.environ.get('DB_LOGIN') or input('DB login: ')
PASSWORD = os.environ.get('DB_PASSWORD') or getpass.getpass('DB password: ')

In [2]:
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

import psycopg2

import pandas as pd



conn = psycopg2.connect(

    host='thomas.proxy.rlwy.net',

    port=51432,

    dbname='academy_db',

    user=LOGIN,

    password=PASSWORD,

    sslmode='require'

)



def sql(sql): return pd.read_sql(sql, conn)



print('Connection established')

Connection established


## 1. Разведка данных: какие таблицы относятся к треку B

### Проверка связи ds_users_2 -> ds_channels (ключевое доказательство)

In [3]:
sql("""

SELECT COUNT(*) FROM ds_users_1;

""")

,count
0,15000


In [4]:
sql("""

SELECT COUNT(*) FROM ds_users_2;

""")

,count
0,8000


In [5]:
sql("""

SELECT COUNT(*) AS matched_channels

FROM ds_users_2 u

JOIN ds_channels c ON u.channel_id = c.channel_id;

""")

,matched_channels
0,8000


In [6]:
sql("""

SELECT COUNT(DISTINCT channel_id) FROM ds_users_2;

""")

,count
0,5


In [7]:
sql("""

SELECT COUNT(*) FROM ds_channels;

""")

,count
0,5


In [8]:
sql("""

SELECT DISTINCT acq_channel FROM ds_users_1;

""")

,acq_channel
0,tiktok_ads
1,google_ads
2,organic
3,instagram_ads
4,referral


In [9]:
sql("""

SELECT DISTINCT channel_name FROM ds_channels;

""")

,channel_name
0,google_ads
1,organic
2,instagram_ads
3,telegram_ads
4,referral


**Итог разведки.** `ds_users_1` (15 000 строк) — не наш трек: значения `acq_channel` там другие (`tiktok_ads` вместо `telegram_ads`), и связи со своим набором каналов нет.
`ds_users_2` (8 000 строк) полностью соединяется с `ds_channels` (5 из 5 каналов совпадают, `matched_channels = 8000`) — это пользователи WalletApp. Дальше работаем с `ds_users_2`, `ds_channels`, `ds_app_events`, `ds_subscriptions`, `ds_plans`, `ds_charges`.

### Базовая воронка (сколько теряется на каждом этапе)

In [10]:
sql("""

WITH funnel AS (

  SELECT user_id,

         MIN(CASE WHEN event_type='signup' THEN event_ts END) AS t_signup,

         MIN(CASE WHEN event_type='kyc_done' THEN event_ts END) AS t_kyc,

         MIN(CASE WHEN event_type='card_linked' THEN event_ts END) AS t_card,

         MIN(CASE WHEN event_type='first_payment' THEN event_ts END) AS t_pay

  FROM ds_app_events

  GROUP BY user_id

)

SELECT

  COUNT(*) AS total_users,

  COUNT(t_signup) AS reached_signup,

  COUNT(t_kyc) AS reached_kyc,

  COUNT(t_card) AS reached_card,

  COUNT(t_pay) AS reached_payment,

  SUM(CASE WHEN t_pay IS NOT NULL AND t_card IS NOT NULL AND t_pay < t_card THEN 1 ELSE 0 END) AS pay_before_card_anomaly,

  SUM(CASE WHEN t_kyc IS NOT NULL AND t_signup IS NOT NULL AND t_kyc < t_signup THEN 1 ELSE 0 END) AS kyc_before_signup_anomaly

FROM funnel;

""")

,total_users,reached_signup,reached_kyc,reached_card,reached_payment,pay_before_card_anomaly,kyc_before_signup_anomaly
0,8000,8000,4918,4918,2247,134,0


**Вывод по базовой воронке.** Из 8 000 зарегистрированных до прохождения KYC доходят 4 918 (61.5%), из них практически все сразу привязывают карту (тоже 4 918 — в этих данных KYC и привязка карты происходят как единый шаг), а до первой оплаты доходят только 2 247 человек — это 45.7% от привязавших карту и всего **28.1% от всех зарегистрированных**. Аномалий в последовательности событий почти нет (`kyc_before_signup_anomaly = 0`, `pay_before_card_anomaly` — всего 134 из 8000, на общую картину не влияет). Значит, деньги теряются в первую очередь на этапе KYC (потеря 39% аудитории) и второй раз — уже после привязки карты, на пути к первой оплате (потеря ещё 54% от дошедших). Разбираем обе точки провала ниже.

## Гипотеза 1 — Каналы привлечения (конверсия + деньги)

In [11]:
sql("""

WITH funnel AS (

  SELECT u.user_id, c.channel_name,

         MIN(CASE WHEN e.event_type='signup' THEN e.event_ts END) AS t_signup,

         MIN(CASE WHEN e.event_type='kyc_done' THEN e.event_ts END) AS t_kyc,

         MIN(CASE WHEN e.event_type='first_payment' THEN e.event_ts END) AS t_pay

  FROM ds_users_2 u

  JOIN ds_channels c ON u.channel_id = c.channel_id

  LEFT JOIN ds_app_events e ON u.user_id = e.user_id

  GROUP BY u.user_id, c.channel_name

)

SELECT channel_name,

       COUNT(*) AS total,

       COUNT(t_kyc) AS reached_kyc,

       ROUND(COUNT(t_kyc)::numeric / COUNT(*) * 100, 1) AS kyc_rate_pct,

       COUNT(t_pay) AS reached_payment,

       ROUND(COUNT(t_pay)::numeric / COUNT(*) * 100, 1) AS payment_rate_pct

FROM funnel

GROUP BY channel_name

ORDER BY kyc_rate_pct DESC;

""")


,channel_name,total,reached_kyc,kyc_rate_pct,reached_payment,payment_rate_pct
0,referral,1199,880,73.4,504,42.0
1,google_ads,1487,956,64.3,458,30.8
2,organic,2375,1467,61.8,678,28.5
3,instagram_ads,1772,1089,61.5,492,27.8
4,telegram_ads,1167,526,45.1,115,9.9


**Что видно по каналам.** `referral` — лидер по всей воронке: 73.4% доходят до KYC и 42.0% становятся платящими. `telegram_ads` — аутсайдер: только 45.1% проходят KYC и всего 9.9% доходят до оплаты — в 4+ раза хуже, чем `referral`. Остальные каналы (`google_ads`, `organic`, `instagram_ads`) держатся в середине, в диапазоне 27.8–30.8% по оплате.

#### Экономика каналов: цена за платящего клиента

In [12]:
sql("""

WITH funnel AS (

  SELECT u.user_id, c.channel_name, c.cost_per_install,

         MIN(CASE WHEN e.event_type='first_payment' THEN e.event_ts END) AS t_pay

  FROM ds_users_2 u

  JOIN ds_channels c ON u.channel_id = c.channel_id

  LEFT JOIN ds_app_events e ON u.user_id = e.user_id

  GROUP BY u.user_id, c.channel_name, c.cost_per_install

)

SELECT channel_name,

       COUNT(*) AS total_users,

       MAX(cost_per_install) AS cost_per_install,

       COUNT(*) * MAX(cost_per_install) AS total_spend,

       COUNT(t_pay) AS paying_users,

       ROUND(COUNT(*) * MAX(cost_per_install)::numeric / NULLIF(COUNT(t_pay),0), 0) AS cost_per_paying_user

FROM funnel

GROUP BY channel_name

ORDER BY cost_per_paying_user DESC;

""")

,channel_name,total_users,cost_per_install,total_spend,paying_users,cost_per_paying_user
0,telegram_ads,1167,9000,10503000,115,91330.0
1,google_ads,1487,22000,32714000,458,71428.0
2,instagram_ads,1772,18000,31896000,492,64829.0
3,referral,1199,3000,3597000,504,7137.0
4,organic,2375,0,0,678,0.0


**Экономика решает.** `organic` и `referral` — самые дешёвые платящие клиенты (0 и 7 137 сум соответственно), а `telegram_ads` — самый дорогой: **91 330 сум за одного платящего пользователя, в 12.8 раза дороже**, чем через `referral`. При этом канал не самый маленький по объёму (1 167 пользователей) — то есть компания стабильно тратит деньги на канал, который почти не окупается.

### Сырые данные для статтеста (экспортировались в CSV для Python)

In [13]:
channel_paid_df = sql("""

SELECT u.user_id, c.channel_name,

    CASE WHEN EXISTS (

        SELECT 1 FROM ds_app_events e 

        WHERE e.user_id = u.user_id AND e.event_type='first_payment'

    ) THEN 1 ELSE 0 END AS paid

FROM ds_users_2 u

JOIN ds_channels c ON u.channel_id = c.channel_id;

""")

channel_paid_df.head()

,user_id,channel_name,paid
0,1,instagram_ads,0
1,2,telegram_ads,0
2,3,organic,0
3,4,organic,0
4,5,google_ads,1


In [14]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(channel_paid_df['channel_name'], channel_paid_df['paid'])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(contingency)
print(f"\nchi2 = {chi2:.2f}, p-value = {p_value:.3g}, dof = {dof}")

paid              0    1
channel_name            
google_ads     1029  458
instagram_ads  1280  492
organic        1697  678
referral        695  504
telegram_ads   1052  115

chi2 = 313.31, p-value = 1.45e-66, dof = 4


Разница между каналами статистически значима (χ² на конверсию канал→оплата), вероятность того, что она возникла случайно — практически нулевая.

### Проверка: одинакова ли демография по каналам (исключаем "плохой таргетинг")

In [15]:
sql("""

SELECT c.channel_name,

       ROUND(AVG(u.age), 1) AS avg_age,

       ROUND(100.0 * SUM(CASE WHEN u.device_os='Android' THEN 1 ELSE 0 END) / COUNT(*), 1) AS android_pct,

       COUNT(*) AS total

FROM ds_users_2 u

JOIN ds_channels c ON u.channel_id = c.channel_id

GROUP BY c.channel_name

ORDER BY c.channel_name;

""")

,channel_name,avg_age,android_pct,total
0,google_ads,29.9,78.5,1487
1,instagram_ads,29.6,79.4,1772
2,organic,29.8,78.0,2375
3,referral,29.6,77.3,1199
4,telegram_ads,29.4,76.2,1167


**Проверка на "плохой таргетинг".** Средний возраст (29.4–29.9 лет) и доля Android (76.2–79.4%) практически одинаковы во всех пяти каналах. Значит, разница в конверсии — не следствие того, что `telegram_ads` приводит другую по составу аудиторию. Дело в качестве/мотивации самого трафика с этого канала.

### Вывод по Гипотезе 1

- Канал привлечения **значимо** влияет на конверсию в оплату (подтверждено статтестом).
- Разница не объясняется составом аудитории — только качеством канала.
- `telegram_ads` — дорогой и неэффективный: 91 330 сум за платящего клиента против 7 137 у `referral` (в 12.8 раза дороже) при вчетверо более низкой конверсии в оплату.
- **Рекомендация:** сократить/остановить закупку `telegram_ads`, перенаправить бюджет на реферальную программу (бонус за приглашение друга).

## Гипотеза 2 — KYC-барьер: зависит ли прохождение KYC от аудитории?

Проверяем, объясняется ли провал на KYC (61.5% в среднем) тем, что определённые сегменты аудитории (возраст, устройство, регион, месяц регистрации) проходят его хуже других.

In [16]:
## Все 4 разреза сразу: device_os, region, age, signup_month

sql("""

WITH funnel AS (

    SELECT 

        u.user_id, 

        u.device_os,

        u.region, 

        u.age,

        DATE_TRUNC('month', u.signup_ts) AS signup_month,

        MIN(CASE WHEN e.event_type='kyc_done' THEN e.event_ts END) AS t_kyc

    FROM ds_users_2 u

    LEFT JOIN ds_app_events e ON u.user_id = e.user_id

    GROUP BY u.user_id, u.device_os, u.region, u.age, u.signup_ts

)

SELECT 'device_os' AS dimension, device_os::text AS segment,

       COUNT(*) AS total, COUNT(t_kyc) AS passed_kyc,

       ROUND(100.0 * COUNT(t_kyc) / COUNT(*), 1) AS kyc_rate_pct

FROM funnel GROUP BY device_os

UNION ALL

SELECT 'region', region::text,

       COUNT(*), COUNT(t_kyc),

       ROUND(100.0 * COUNT(t_kyc) / COUNT(*), 1)

FROM funnel GROUP BY region

UNION ALL

SELECT 'age_group',

       CASE 

         WHEN age < 25 THEN '18-24'

         WHEN age < 35 THEN '25-34'

         WHEN age < 45 THEN '35-44'

         WHEN age < 55 THEN '45-54'

         ELSE '55+'

       END,

       COUNT(*), COUNT(t_kyc),

       ROUND(100.0 * COUNT(t_kyc) / COUNT(*), 1)

FROM funnel GROUP BY 2

UNION ALL

SELECT 'signup_month', signup_month::text,

       COUNT(*), COUNT(t_kyc),

       ROUND(100.0 * COUNT(t_kyc) / COUNT(*), 1)

FROM funnel GROUP BY signup_month

ORDER BY dimension, segment;

""")

,dimension,segment,total,passed_kyc,kyc_rate_pct
0,age_group,18-24,2338,1432,61.2
1,age_group,25-34,3403,2120,62.3
2,age_group,35-44,1876,1143,60.9
3,age_group,45-54,363,212,58.4
4,age_group,55+,20,11,55.0
5,device_os,Android,6243,3820,61.2
6,device_os,iOS,1757,1098,62.5
7,region,Andijan,814,505,62.0
8,region,Bukhara,809,482,59.6
9,region,Fergana,1026,634,61.8


**Вывод по Гипотезе 2.** Разброс KYC-конверсии по всем срезам укладывается в узкий диапазон **55–64%**, без выраженного тренда:
- по устройству — Android 61.2% vs iOS 62.5% (разница на уровне погрешности);
- по региону — от 59.6% (Бухара) до 63.4% (Нукус), разброс ~4 п.п.;
- по возрасту — плавное снижение с 61.2% (18-24) до 55.0% (55+), но группа 55+ — это всего 20 человек, статистически ненадёжно;
- по месяцу регистрации — колебания 57.5–64.3% без устойчивого роста или падения во времени.

Ни один срез не даёт разрыва, сравнимого с разрывом между каналами (12–73%). **Значит, гипотеза не подтвердилась**: провал на KYC — не следствие того, кто именно регистрируется, а системная проблема самого процесса (скорее всего, трение в самой форме/шагах верификации), которая одинаково задевает всех. Это меняет рекомендацию: не таргетировать конкретный сегмент, а упрощать сам KYC-флоу для всех пользователей.

### Отток по тарифам — проверяем, не "жалко ли денег" на дорогих планах

In [17]:
sql("""

SELECT p.plan_name, p.monthly_price,

       COUNT(*) AS total_subs,

       SUM(CASE WHEN s.status='canceled' THEN 1 ELSE 0 END) AS canceled,

       ROUND(100.0 * SUM(CASE WHEN s.status='canceled' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct

FROM ds_subscriptions s

JOIN ds_plans p ON s.plan_id = p.plan_id

GROUP BY p.plan_name, p.monthly_price

ORDER BY p.monthly_price;

""")

,plan_name,monthly_price,total_subs,canceled,churn_rate_pct
0,Lite,19000,749,160,21.4
1,Plus,49000,681,141,20.7
2,Premium,99000,229,42,18.3


**Не подтвердилось.** Отток почти одинаковый на всех тарифах: Lite 21.4%, Plus 20.7%, Premium 18.3% — разброс всего 3 п.п., и он даже в обратную сторону (самый дорогой тариф отменяют чуть реже, а не чаще). Цена подписки — не причина отмены.

### Скорость активации → оплата — оказалось не значимо

In [18]:
sql("""

WITH funnel AS (

  SELECT user_id,

         MIN(CASE WHEN event_type='signup' THEN event_ts END) AS t_signup,

         MIN(CASE WHEN event_type='card_linked' THEN event_ts END) AS t_card,

         MIN(CASE WHEN event_type='first_payment' THEN event_ts END) AS t_pay

  FROM ds_app_events

  GROUP BY user_id

),

speed AS (

  SELECT user_id,

         EXTRACT(EPOCH FROM (t_card - t_signup))/3600 AS hours_to_card,

         t_pay IS NOT NULL AS paid

  FROM funnel

  WHERE t_card IS NOT NULL

)

SELECT

  CASE 

    WHEN hours_to_card <= 24 THEN '0-24ч'

    WHEN hours_to_card <= 72 THEN '1-3 дня'

    WHEN hours_to_card <= 168 THEN '3-7 дней'

    ELSE '7+ дней'

  END AS speed_group,

  COUNT(*) AS total,

  SUM(CASE WHEN paid THEN 1 ELSE 0 END) AS paid_count,

  ROUND(100.0 * SUM(CASE WHEN paid THEN 1 ELSE 0 END) / COUNT(*), 1) AS payment_rate_pct

FROM speed

GROUP BY speed_group

ORDER BY MIN(hours_to_card);

""")

,speed_group,total,paid_count,payment_rate_pct
0,0-24ч,956,430,45.0
1,1-3 дня,1944,909,46.8
2,3-7 дней,2018,908,45.0


**Тоже не подтвердилось.** Скорость привязки карты после регистрации почти не влияет на то, дойдёт ли человек до оплаты: 45.0% (0-24ч), 46.8% (1-3 дня), 45.0% (3-7 дней) — разница в пределах шума.

### Failed charges по тарифам — ставка отказов одинакова везде (~5.5%)

In [19]:
sql("""

SELECT p.plan_name, p.monthly_price,

       COUNT(*) AS total_charges,

       SUM(CASE WHEN c.status='failed' THEN 1 ELSE 0 END) AS failed,

       ROUND(100.0 * SUM(CASE WHEN c.status='failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failed_rate_pct,

       SUM(CASE WHEN c.status='failed' THEN c.amount ELSE 0 END) AS lost_revenue

FROM ds_charges c

JOIN ds_subscriptions s ON c.subscription_id = s.subscription_id

JOIN ds_plans p ON s.plan_id = p.plan_id

GROUP BY p.plan_name, p.monthly_price

ORDER BY p.monthly_price;

""")

,plan_name,monthly_price,total_charges,failed,failed_rate_pct,lost_revenue
0,Lite,19000,5994,334,5.6,6346000.0
1,Plus,49000,5400,312,5.8,15288000.0
2,Premium,99000,1695,91,5.4,9009000.0


**Ставка отказов списаний одинакова везде** — 5.4-5.8% по всем тарифам, зависимости от цены плана нет. Но сама по себе эта проблема — реальные потери в деньгах (см. ниже).

### Итог по вторичным проверкам

Ни цена тарифа, ни скорость активации, ни тариф как таковой не объясняют отток или сбои платежей — эти явления равномерны по всей базе. Значит, для управления оттоком и сбоями нужны меры уровня всего продукта (grace period при сбое оплаты, упрощение KYC), а не сегментированные под конкретный тариф или профиль пользователя.

## 2. Оценка финансового эффекта

Переходим от "где теряются пользователи" к "сколько это стоит компании в деньгах" — по каждой из найденных точек провала.

### Упущенная выручка от KYC-барьера

In [20]:
sql("""

WITH funnel_counts AS (

    SELECT 

        COUNT(DISTINCT CASE WHEN event_type='signup' THEN user_id END) AS total_signup,

        COUNT(DISTINCT CASE WHEN event_type='kyc_done' THEN user_id END) AS total_kyc

    FROM ds_app_events

),

avg_rev AS (

    SELECT AVG(amount) AS avg_charge

    FROM ds_charges

    WHERE status = 'succeeded'

)

SELECT 

    f.total_signup,

    f.total_kyc,

    f.total_signup - f.total_kyc AS lost_at_kyc_stage,

    ROUND(a.avg_charge, 0) AS avg_charge_amount,

    ROUND((f.total_signup - f.total_kyc) * a.avg_charge, 0) AS potential_lost_revenue

FROM funnel_counts f, avg_rev a;

""")

,total_signup,total_kyc,lost_at_kyc_stage,avg_charge_amount,potential_lost_revenue
0,8000,4918,3082,41746.0,128661523.0


3 082 человека (39% от всех регистраций) не проходят KYC. Если бы они конвертировались в оплату на среднем уровне (41 746 сум/платёж), упущенная выручка оценивается в **~128.7 млн сум**. Это верхняя, оценочная планка (не все 3082 обязательно стали бы платящими), но масштаб показывает, почему упрощение KYC — по одной этой причине уже приоритетная задача.

### Реальные потери от неудачных списаний

In [21]:
sql("""

SELECT 

    COUNT(*) AS failed_charges_count,

    ROUND(AVG(amount), 0) AS avg_failed_amount,

    SUM(amount) AS total_lost_revenue

FROM ds_charges

WHERE status = 'failed';

""")

,failed_charges_count,avg_failed_amount,total_lost_revenue
0,737,41578.0,30643000.0


А это уже не оценка, а прямые потери: 737 неудачных списаний на сумму **~30.6 млн сум** — деньги, которые компания не получила от уже привлечённых платящих клиентов из-за технического сбоя оплаты (не хватило средств на карте, карта просрочена и т.п.).

### Недополученный LTV от отменённых подписок

In [22]:
sql("""

WITH avg_lifetime AS (

    SELECT AVG(EXTRACT(EPOCH FROM (COALESCE(canceled_at, NOW()) - started_at))/2592000) AS avg_months_active

    FROM ds_subscriptions

    WHERE status = 'active'

),

canceled_summary AS (

    SELECT s.subscription_id, p.monthly_price,

           EXTRACT(EPOCH FROM (s.canceled_at - s.started_at))/2592000 AS months_before_cancel

    FROM ds_subscriptions s

    JOIN ds_plans p ON s.plan_id = p.plan_id

    WHERE s.status = 'canceled'

)

SELECT 

    COUNT(*) AS canceled_subs,

    ROUND(AVG(months_before_cancel), 1) AS avg_months_before_cancel,

    ROUND(SUM(monthly_price * months_before_cancel), 0) AS revenue_earned_before_cancel,

    (SELECT ROUND(avg_months_active::numeric, 1) FROM avg_lifetime) AS avg_months_if_stayed

FROM canceled_summary;

""")

,canceled_subs,avg_months_before_cancel,revenue_earned_before_cancel,avg_months_if_stayed
0,343,5.0,71305000.0,8.1


343 отменённые подписки в среднем "прожили" **5 месяцев** и принесли **~71.3 млн сум**, тогда как подписки, которые остаются активными, в среднем держатся **8 месяцев**. Разница в 3 месяца на каждую из 343 отменённых подписок — это дополнительная упущенная выручка, которую можно вернуть, если снизить преждевременный отток (например, за счёт grace period при сбое оплаты, раз именно сбои — наиболее вероятный триггер части отмен).

### Оконная функция: динамика новых подписок по месяцам (running total + MoM)

Смотрим, растёт ли база подписчиков во времени и как накопленный отток соотносится с ростом — с помощью `SUM() OVER` (кумулятивный итог) и `LAG() OVER` (прирост месяц к месяцу).

In [23]:
sql("""
WITH monthly_subs AS (
    SELECT DATE_TRUNC('month', started_at) AS month,
           COUNT(*) AS new_subs,
           SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) AS canceled_from_cohort
    FROM ds_subscriptions
    GROUP BY 1
)
SELECT
    month,
    new_subs,
    SUM(new_subs) OVER (ORDER BY month) AS cumulative_subs,
    LAG(new_subs) OVER (ORDER BY month) AS prev_month_new_subs,
    ROUND(
        100.0 * (new_subs - LAG(new_subs) OVER (ORDER BY month))
        / NULLIF(LAG(new_subs) OVER (ORDER BY month), 0), 1
    ) AS mom_growth_pct,
    canceled_from_cohort,
    ROUND(
        100.0 * SUM(canceled_from_cohort) OVER (ORDER BY month)
        / SUM(new_subs) OVER (ORDER BY month), 1
    ) AS cumulative_churn_pct
FROM monthly_subs
ORDER BY month;
""")

,month,new_subs,cumulative_subs,prev_month_new_subs,mom_growth_pct,canceled_from_cohort,cumulative_churn_pct
0,2025-01-01,45,45.0,NaN,NaN,12,26.7
1,2025-02-01,52,97.0,45.0,15.6,15,27.8
2,2025-03-01,77,174.0,52.0,48.1,27,31.0
3,2025-04-01,75,249.0,77.0,-2.6,29,33.3
4,2025-05-01,99,348.0,75.0,32.0,37,34.5
5,2025-06-01,85,433.0,99.0,-14.1,26,33.7
6,2025-07-01,88,521.0,85.0,3.5,28,33.4
7,2025-08-01,99,620.0,88.0,12.5,27,32.4
8,2025-09-01,79,699.0,99.0,-20.2,23,32.0
9,2025-10-01,106,805.0,79.0,34.2,21,30.4


## Общие выводы и рекомендации

**Находка 1 — канал привлечения.** `telegram_ads` даёт вчетверо более низкую конверсию в оплату, чем `referral`, при этом платящий клиент оттуда обходится в 12.8 раза дороже. Разница статистически значима и не объясняется составом аудитории. → Сократить/остановить `telegram_ads`, усилить реферальную программу.

**Находка 2 — KYC-барьер.** 39% пользователей не проходят KYC, и это происходит **равномерно** по всем сегментам (возраст, устройство, регион, период регистрации) — не таргетинг виноват, а сам процесс верификации. Потенциально упущено ~128.7 млн сум. → Упрощать KYC-флоу для всех, а не для отдельного сегмента.

**Побочные находки:** цена тарифа, скорость активации карты и тариф как таковой не влияют ни на отток, ни на сбои оплаты — эти проблемы равномерны по базе. Прямые потери от сбоев списания — ~30.6 млн сум; отменённые подписки не дожили в среднем 3 месяца до срока активных — упущенный LTV.

**Чего не хватило:** данных о том, на каком именно шаге KYC-формы пользователь останавливается (сама форма верификации не размечена событиями по шагам) — следующий шаг для продуктовой команды.

In [24]:
conn.close()

print('Connection closed')

Connection closed
